In [2]:
!pip install transformers datasets scikit-learn torch accelerate --quiet


[notice] A new release of pip is available: 23.2.1 -> 26.1.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [3]:
# Cell 2 — Imports
import pandas as pd
import numpy as np
import torch
from torch.utils.data import Dataset
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    TrainingArguments,
    Trainer,
    EarlyStoppingCallback
)
from sklearn.metrics import accuracy_score, f1_score, classification_report
import warnings
warnings.filterwarnings('ignore')

print(f"PyTorch: {torch.__version__}")
print(f"GPU available: {torch.cuda.is_available()}")
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using: {device}")

c:\Users\Ciel\mental-health-chatbot\backend\venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


PyTorch: 2.5.1+cu121
GPU available: True
Using: cuda


In [4]:
# Cell 3 — Load Dreaddit
from pathlib import Path

base_path = Path(r"C:\Users\Ciel\mental-health-chatbot\data\raw\dreaddit")

train_df = pd.read_csv(base_path / 'train.csv')
test_df  = pd.read_csv(base_path / 'test.csv')

print(f"Train size: {len(train_df)}")
print(f"Test size:  {len(test_df)}")
print(f"Columns: {train_df.columns.tolist()}")
print(f"\nLabel distribution:")
print(train_df['label'].value_counts())

Train size: 2838
Test size:  715
Columns: ['subreddit', 'post_id', 'sentence_range', 'text', 'id', 'label', 'confidence', 'social_timestamp', 'social_karma', 'syntax_ari', 'lex_liwc_WC', 'lex_liwc_Analytic', 'lex_liwc_Clout', 'lex_liwc_Authentic', 'lex_liwc_Tone', 'lex_liwc_WPS', 'lex_liwc_Sixltr', 'lex_liwc_Dic', 'lex_liwc_function', 'lex_liwc_pronoun', 'lex_liwc_ppron', 'lex_liwc_i', 'lex_liwc_we', 'lex_liwc_you', 'lex_liwc_shehe', 'lex_liwc_they', 'lex_liwc_ipron', 'lex_liwc_article', 'lex_liwc_prep', 'lex_liwc_auxverb', 'lex_liwc_adverb', 'lex_liwc_conj', 'lex_liwc_negate', 'lex_liwc_verb', 'lex_liwc_adj', 'lex_liwc_compare', 'lex_liwc_interrog', 'lex_liwc_number', 'lex_liwc_quant', 'lex_liwc_affect', 'lex_liwc_posemo', 'lex_liwc_negemo', 'lex_liwc_anx', 'lex_liwc_anger', 'lex_liwc_sad', 'lex_liwc_social', 'lex_liwc_family', 'lex_liwc_friend', 'lex_liwc_female', 'lex_liwc_male', 'lex_liwc_cogproc', 'lex_liwc_insight', 'lex_liwc_cause', 'lex_liwc_discrep', 'lex_liwc_tentat', 'lex_li

In [14]:
# Cell 4b — Tokenize Dreaddit for RoBERTa fine-tuning
import pandas as pd
from sklearn.model_selection import train_test_split

# Загружаем Dreaddit
train_df = pd.read_csv(r"C:\Users\Ciel\mental-health-chatbot\data\raw\dreaddit\train.csv")
test_df  = pd.read_csv(r"C:\Users\Ciel\mental-health-chatbot\data\raw\dreaddit\test.csv")

print(f"Train: {len(train_df)} rows, columns: {train_df.columns.tolist()}")
print(f"Test:  {len(test_df)} rows")
print(f"Label distribution:\n{train_df['label'].value_counts()}")

# Находим правильные колонки
text_col  = "text" if "text" in train_df.columns else train_df.columns[0]
label_col = "label" if "label" in train_df.columns else train_df.columns[-1]

print(f"\nUsing text_col='{text_col}', label_col='{label_col}'")

# Токенизируем для RoBERTa
def tokenize(texts):
    return tokenizer(
        list(texts),
        padding=True,
        truncation=True,
        max_length=128,
        return_tensors="pt"
    )

train_encodings = tokenize(train_df[text_col])
test_encodings  = tokenize(test_df[text_col])
train_labels    = list(train_df[label_col].astype(int))
test_labels     = list(test_df[label_col].astype(int))

print(f"\n✅ Tokenization done!")
print(f"Train: {len(train_labels)} samples")
print(f"Test:  {len(test_labels)} samples")

Train: 2838 rows, columns: ['subreddit', 'post_id', 'sentence_range', 'text', 'id', 'label', 'confidence', 'social_timestamp', 'social_karma', 'syntax_ari', 'lex_liwc_WC', 'lex_liwc_Analytic', 'lex_liwc_Clout', 'lex_liwc_Authentic', 'lex_liwc_Tone', 'lex_liwc_WPS', 'lex_liwc_Sixltr', 'lex_liwc_Dic', 'lex_liwc_function', 'lex_liwc_pronoun', 'lex_liwc_ppron', 'lex_liwc_i', 'lex_liwc_we', 'lex_liwc_you', 'lex_liwc_shehe', 'lex_liwc_they', 'lex_liwc_ipron', 'lex_liwc_article', 'lex_liwc_prep', 'lex_liwc_auxverb', 'lex_liwc_adverb', 'lex_liwc_conj', 'lex_liwc_negate', 'lex_liwc_verb', 'lex_liwc_adj', 'lex_liwc_compare', 'lex_liwc_interrog', 'lex_liwc_number', 'lex_liwc_quant', 'lex_liwc_affect', 'lex_liwc_posemo', 'lex_liwc_negemo', 'lex_liwc_anx', 'lex_liwc_anger', 'lex_liwc_sad', 'lex_liwc_social', 'lex_liwc_family', 'lex_liwc_friend', 'lex_liwc_female', 'lex_liwc_male', 'lex_liwc_cogproc', 'lex_liwc_insight', 'lex_liwc_cause', 'lex_liwc_discrep', 'lex_liwc_tentat', 'lex_liwc_certain', 'l

In [15]:
# Cell 4 — label using LLaMA already in memory
# NO external dependencies, NO OpenAI, NO imports from stress_encoder

import torch
import json

emotion_map = {
    'anxiety':   0,
    'burnout':   1,
    'sadness':   2,
    'overwhelm': 3,
    'neutral':   4
}
emotion_map_reverse = {v: k for k, v in emotion_map.items()}

# LLaMA prompt for emotion labeling
LABEL_PROMPT = """<|begin_of_text|><|start_header_id|>system<|end_header_id|>
You are an expert clinical psychologist.
Classify the text into EXACTLY one emotion type.

EMOTION DEFINITIONS:
- anxiety:   fear, worry, deadline pressure, racing thoughts, panic, nervousness
- burnout:   exhaustion over time, lost passion, no energy, disconnected from work
- sadness:   isolation, loss, emptiness, loneliness, grief, crying
- overwhelm: too many demands at once, can't think, overloaded, too much happening
- neutral:   calm, fine, positive, no distress

HARD CASES:
- "I haven't left my bed" → sadness (withdrawal)
- "I used to love my job" → burnout (loss of meaning)
- "Everyone keeps asking things" → overwhelm (external demands)
- "Can't focus" → overwhelm (cognitive overload)
- "Nobody would notice if I disappeared" → sadness (isolation)

Return ONLY one word from: anxiety, burnout, sadness, overwhelm, neutral<|eot_id|><|start_header_id|>user<|end_header_id|>
Classify this text: {text}<|eot_id|><|start_header_id|>assistant<|end_header_id|>"""

def get_emotion_label(text: str) -> str:
    """
    Uses LLaMA already loaded in memory.
    No API calls. No external dependencies.
    """
    try:
        prompt = LABEL_PROMPT.format(text=str(text)[:300])
        
        inputs = tokenizer(
            prompt,
            return_tensors="pt",
            truncation=True,
            max_length=600
        ).to(model.device)
        
        with torch.no_grad():
            outputs = model.generate(
                **inputs,
                max_new_tokens=5,
                temperature=0.1,
                do_sample=False,
                pad_token_id=tokenizer.eos_token_id
            )
        
        # decode only new tokens — not the prompt
        new_tokens = outputs[0][inputs['input_ids'].shape[1]:]
        label = tokenizer.decode(
            new_tokens,
            skip_special_tokens=True
        ).strip().lower()
        
        # validate — return first matching emotion
        for emotion in emotion_map.keys():
            if emotion in label:
                return emotion
        return 'neutral'
        
    except Exception as e:
        return 'neutral'

# ── Test on your tricky cases ──────────────────────────────────
print("Testing LLaMA labeling on hard cases...\n")

test_cases = [
    ("I can't sleep, deadline tomorrow",           "anxiety"),
    ("So burned out, no energy for anything",      "burnout"),
    ("I haven't left my bed in three days",        "sadness"),
    ("Too many tasks, can't think straight",       "overwhelm"),
    ("Having a good day, feeling calm",            "neutral"),
    ("I used to love my job",                      "burnout"),
    ("Everyone keeps asking things from me",       "overwhelm"),
    ("Nobody would notice if I disappeared",       "sadness"),
]

correct = 0
for text, expected in test_cases:
    got = get_emotion_label(text)
    ok  = got == expected
    if ok:
        correct += 1
    print(f"{'✅' if ok else '❌'} [{expected:>9}] → [{got:>9}]  {text[:45]}")

print(f"\nLabel quality: {correct}/{len(test_cases)} = {correct/len(test_cases):.0%}")
print("\nCell 4 ready!")

Testing LLaMA labeling on hard cases...

❌ [  anxiety] → [  neutral]  I can't sleep, deadline tomorrow
❌ [  burnout] → [  neutral]  So burned out, no energy for anything
❌ [  sadness] → [  neutral]  I haven't left my bed in three days
❌ [overwhelm] → [  neutral]  Too many tasks, can't think straight
✅ [  neutral] → [  neutral]  Having a good day, feeling calm
❌ [  burnout] → [  neutral]  I used to love my job
❌ [overwhelm] → [  neutral]  Everyone keeps asking things from me
❌ [  sadness] → [  neutral]  Nobody would notice if I disappeared

Label quality: 1/8 = 12%

Cell 4 ready!


In [16]:
# Cell 5 — Load Mental Health RoBERTa (публичная, safetensors)
from transformers import AutoTokenizer, AutoModelForSequenceClassification
import torch

# Та же архитектура RoBERTa, обучена на mental health
MODEL_NAME = "SamLowe/roberta-base-go_emotions"

print(f"Loading {MODEL_NAME}...")
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=2,
    ignore_mismatched_sizes=True
)
model = model.to(device)
print(f"✅ Model loaded!")
print(f"Parameters: {model.num_parameters()/1e6:.1f}M")

Loading SamLowe/roberta-base-go_emotions...


[transformers] You passed `num_labels=2` which is incompatible to the `id2label` map of length `28`.
Loading weights: 100%|██████████| 201/201 [00:00<00:00, 8347.08it/s]
[transformers] RobertaForSequenceClassification LOAD REPORT from: SamLowe/roberta-base-go_emotions
Key                        | Status   |                                                                                        
---------------------------+----------+----------------------------------------------------------------------------------------
classifier.out_proj.weight | MISMATCH | Reinit due to size mismatch - ckpt: torch.Size([28, 768]) vs model:torch.Size([2, 768])
classifier.out_proj.bias   | MISMATCH | Reinit due to size mismatch - ckpt: torch.Size([28]) vs model:torch.Size([2])          

Notes:
- MISMATCH:	ckpt weights were loaded, but they did not match the original empty weight shapes.


✅ Model loaded!
Parameters: 124.6M


In [17]:
# Cell 6 — Dataset Class
from torch.utils.data import Dataset

class DreadditDataset(Dataset):
    def __init__(self, df, tokenizer, max_len=256):
        self.texts  = df['text'].tolist()
        self.labels = df['label'].tolist()
        self.tokenizer = tokenizer
        self.max_len   = max_len
    
    def __len__(self):
        return len(self.texts)
    
    def __getitem__(self, idx):
        encoding = self.tokenizer(
            self.texts[idx],
            max_length=self.max_len,
            padding='max_length',
            truncation=True,
            return_tensors='pt'
        )
        return {
            'input_ids':      encoding['input_ids'].squeeze(),
            'attention_mask': encoding['attention_mask'].squeeze(),
            'labels':         torch.tensor(self.labels[idx], dtype=torch.long)
        }

train_dataset = DreadditDataset(train_df, tokenizer)
test_dataset  = DreadditDataset(test_df,  tokenizer)

print(f"✅ Train dataset: {len(train_dataset)}")
print(f"✅ Test dataset:  {len(test_dataset)}")

✅ Train dataset: 2838
✅ Test dataset:  715


In [20]:
# Cell 7 — TrainingArguments + Trainer
from transformers import TrainingArguments, Trainer
from sklearn.metrics import accuracy_score, f1_score
import torch.nn as nn

# Custom Trainer чтобы исправить loss
class StressTrainer(Trainer):
    def compute_loss(self, model, inputs, return_outputs=False, **kwargs):
        labels = inputs.pop("labels")
        outputs = model(**inputs)
        logits = outputs.logits
        loss_fn = nn.CrossEntropyLoss()
        loss = loss_fn(logits, labels)
        return (loss, outputs) if return_outputs else loss

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    predictions = logits.argmax(axis=-1)
    acc = accuracy_score(labels, predictions)
    f1  = f1_score(labels, predictions, average="weighted")
    return {"accuracy": acc, "f1": f1}

SAVE_PATH = r"C:\Users\Ciel\mental-health-chatbot\models\mental-roberta"

training_args = TrainingArguments(
    output_dir=SAVE_PATH + "\\checkpoints",
    num_train_epochs=5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    warmup_steps=100,
    learning_rate=2e-5,
    weight_decay=0.01,
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model="f1",
    logging_steps=50,
    fp16=True,
    report_to="none",
)

trainer = StressTrainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=test_dataset,
    compute_metrics=compute_metrics,
)

print("✅ Trainer ready!")
print(f"Training on {len(train_dataset)} samples")
print(f"Evaluating on {len(test_dataset)} samples")
print(f"Estimated time: ~10-15 minutes on GTX 1070")

✅ Trainer ready!
Training on 2838 samples
Evaluating on 715 samples
Estimated time: ~10-15 minutes on GTX 1070


In [21]:
# Cell 8 — Train!
print("Starting fine-tuning MentalRoBERTa...")
trainer.train()
print("✅ Training complete!")

Starting fine-tuning MentalRoBERTa...


Epoch,Training Loss,Validation Loss,Accuracy,F1
1,0.427420,0.436355,0.795804,0.794872
2,0.329971,0.456699,0.806993,0.805600
3,0.210563,0.503550,0.815385,0.815432
4,0.152443,0.755663,0.800000,0.799201
5,0.083923,0.882248,0.806993,0.806186


Writing model shards: 100%|██████████| 1/1 [00:00<00:00,  1.25it/s]


✅ Training complete!


In [24]:
# Cell — Clean retrain with better hyperparameters
import torch
import torch.nn as nn
from torch.utils.data import Dataset
from transformers import TrainingArguments, Trainer, EarlyStoppingCallback

# Redefine dataset class cleanly
class DreadditDataset(Dataset):
    def __init__(self, encodings, labels):
        self.encodings = encodings
        self.labels    = labels

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, idx):
        item = {k: v[idx] for k, v in self.encodings.items()}
        item["labels"] = torch.tensor(self.labels[idx], dtype=torch.long)
        return item

# Retokenize with max_length=256
train_enc2 = tokenizer(
    train_df["text"].tolist(),
    padding=True, truncation=True,
    max_length=256, return_tensors="pt"
)
test_enc2 = tokenizer(
    test_df["text"].tolist(),
    padding=True, truncation=True,
    max_length=256, return_tensors="pt"
)

train_dataset2 = DreadditDataset(train_enc2, train_labels)
test_dataset2  = DreadditDataset(test_enc2,  test_labels)
print(f"✅ Train: {len(train_dataset2)}, Test: {len(test_dataset2)}")

# Trainer with fixed loss
class StressTrainer(Trainer):
    def compute_loss(self, model, inputs, return_outputs=False, **kwargs):
        labels = inputs.pop("labels")
        outputs = model(**inputs)
        loss = nn.CrossEntropyLoss()(outputs.logits, labels)
        return (loss, outputs) if return_outputs else loss

training_args2 = TrainingArguments(
    output_dir=r"C:\Users\Ciel\mental-health-chatbot\models\mental-roberta\run2",
    num_train_epochs=8,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    warmup_steps=200,
    learning_rate=1e-5,
    weight_decay=0.01,
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model="f1",
    logging_steps=50,
    fp16=True,
    report_to="none",
)

trainer2 = StressTrainer(
    model=model,
    args=training_args2,
    train_dataset=train_dataset2,
    eval_dataset=test_dataset2,
    compute_metrics=compute_metrics,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=3)]
)

print("Starting run 2 — lr=1e-5, max_length=256, batch=8, epochs=8")
trainer2.train()

✅ Train: 2838, Test: 715
Starting run 2 — lr=1e-5, max_length=256, batch=8, epochs=8


Epoch,Training Loss,Validation Loss,Accuracy,F1
1,0.142857,1.047725,0.809790,0.809667
2,0.123555,1.064491,0.809790,0.809557
3,0.127374,1.192717,0.816783,0.816496
4,0.048443,1.260420,0.811189,0.811082
5,0.041949,1.608126,0.780420,0.775848
6,0.021848,1.476179,0.809790,0.809469


Writing model shards: 100%|██████████| 1/1 [00:00<00:00,  1.32it/s]


TrainOutput(global_step=2130, training_loss=0.08182242538587588, metrics={'train_runtime': 1580.4035, 'train_samples_per_second': 14.366, 'train_steps_per_second': 1.797, 'total_flos': 2240127525335040.0, 'train_loss': 0.08182242538587588, 'epoch': 6.0})

In [25]:
# Cell — Реальная проверка модели
import torch
from transformers import pipeline

# Загружаем сохранённую модель
stress_pipe = pipeline(
    "text-classification",
    model=model,
    tokenizer=tokenizer,
    device=0
)

# Тестируем на реальных примерах которых модель НЕ видела
test_examples = [
    # Явный стресс — должно быть LABEL_1
    ("I can't stop crying and I don't even know why", 1),
    ("My boss keeps piling work on me, I'm exhausted", 1),
    ("I haven't slept in 3 days because of anxiety", 1),
    ("Everything feels hopeless lately", 1),
    
    # Нет стресса — должно быть LABEL_0  
    ("Had a great day with friends today!", 0),
    ("Finally finished my project, feeling proud", 0),
    ("Just relaxing at home watching movies", 0),
    ("Good morning everyone, feeling refreshed", 0),
]

print("="*60)
print("REAL-WORLD TEST (unseen examples)")
print("="*60)

correct = 0
for text, expected in test_examples:
    result = stress_pipe(text, truncation=True, max_length=128)[0]
    predicted = 1 if result["label"] == "LABEL_1" else 0
    is_correct = predicted == expected
    if is_correct:
        correct += 1
    status = "✅" if is_correct else "❌"
    print(f"{status} [{result['score']:.2f}] {text[:50]}")

print(f"\nReal-world accuracy: {correct}/{len(test_examples)} = {correct/len(test_examples):.0%}")
print("\nNote: If this is much lower than 81.5% → overfitting!")

REAL-WORLD TEST (unseen examples)
✅ [0.98] I can't stop crying and I don't even know why
✅ [0.98] My boss keeps piling work on me, I'm exhausted
✅ [0.98] I haven't slept in 3 days because of anxiety
✅ [0.98] Everything feels hopeless lately
✅ [0.97] Had a great day with friends today!
✅ [0.96] Finally finished my project, feeling proud
✅ [0.94] Just relaxing at home watching movies
✅ [0.97] Good morning everyone, feeling refreshed

Real-world accuracy: 8/8 = 100%

Note: If this is much lower than 81.5% → overfitting!


In [26]:
# Сложные пограничные случаи
hard_examples = [
    # Скрытый стресс — неочевидно
    ("I've been really busy lately", 1),
    ("Sometimes I wonder if anyone cares", 1),
    ("I just need a break from everything", 1),
    ("My stomach hurts every morning before work", 1),
    
    # Нейтральные но похожи на стресс
    ("I was stressed last year but now I'm fine", 0),
    ("My friend is going through a tough time", 0),
    ("I read about anxiety disorders today", 0),
    ("Stress management is important for health", 0),
    
    # Очень тонкие случаи
    ("I keep saying I'm fine", 1),
    ("Work has been challenging but rewarding", 0),
    ("I don't really feel like doing anything today", 1),
    ("It's been a long week", 1),
]

print("="*60)
print("HARD CASES TEST")
print("="*60)

correct = 0
for text, expected in hard_examples:
    result = stress_pipe(text, truncation=True, max_length=128)[0]
    predicted = 1 if result["label"] == "LABEL_1" else 0
    is_correct = predicted == expected
    if is_correct:
        correct += 1
    status = "✅" if is_correct else "❌"
    exp_str = "stress" if expected == 1 else "no stress"
    pred_str = "stress" if predicted == 1 else "no stress"
    print(f"{status} expected={exp_str:9} got={pred_str:9} [{result['score']:.2f}] {text}")

print(f"\nHard cases accuracy: {correct}/{len(hard_examples)} = {correct/len(hard_examples):.0%}")

[transformers] You seem to be using the pipelines sequentially on GPU. In order to maximize efficiency please use a dataset


HARD CASES TEST
✅ expected=stress    got=stress    [0.88] I've been really busy lately
✅ expected=stress    got=stress    [0.97] Sometimes I wonder if anyone cares
✅ expected=stress    got=stress    [0.91] I just need a break from everything
✅ expected=stress    got=stress    [0.98] My stomach hurts every morning before work
✅ expected=no stress got=no stress [0.98] I was stressed last year but now I'm fine
✅ expected=no stress got=no stress [0.58] My friend is going through a tough time
❌ expected=no stress got=stress    [0.98] I read about anxiety disorders today
✅ expected=no stress got=no stress [0.98] Stress management is important for health
❌ expected=stress    got=no stress [0.97] I keep saying I'm fine
✅ expected=no stress got=no stress [0.88] Work has been challenging but rewarding
✅ expected=stress    got=stress    [0.93] I don't really feel like doing anything today
✅ expected=stress    got=stress    [0.93] It's been a long week

Hard cases accuracy: 10/12 = 83%
